# Gaze Data in NLP Research: Recording Methods and Analysis

**NLPAICS 2026 Summer School — The Paradigm Shift** · Day 3 · Wednesday 17 June 2026

**Lecturer:** Cengiz Acartürk

> Before running anything, make sure the kernel is **NLPAICS 08** (menu: *Kernel → Change Kernel*). It should already be selected.

## 0 · Environment check

In [ ]:
# --- Environment check: run this cell first ---------------------------------
# It verifies you are on this lesson's kernel and that the GPU is visible.
import sys

assert ".venv" in sys.executable, (
    "Wrong kernel! In the menu choose: Kernel > Change Kernel > 'NLPAICS 08"
)
print("Kernel OK:", sys.executable)

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed (fine if this lesson doesn't need it)")

In [ ]:
# --- MILESTONE 1: Read and Modify Human Eye-Tracking Data from the GECO Corpus ---
import pandas as pd

# Load the dataset
df_full = pd.read_csv("geco_subset.csv")

# Extract the first sentence of the novel (rows 0 through 21) --- Agatha Christie’s The Mysterious Affair at Styles
df_human = df_full.head(22).copy()

# Skipped Words:
# Readers skip some words! We may fill 'NaN' reading times with 0 milliseconds. 
# A better option might be to leave than as NaN, but we will continue by filling 0 ms for now.
df_human['WORD_TOTAL_READING_TIME'] = df_human['WORD_TOTAL_READING_TIME'].fillna(0)

# Display the sentence we are analyzing
sentence = " ".join(df_human['WORD'].astype(str).tolist())
print(f"Analyzing Sentence: {sentence}")
df_human[['WORD', 'WORD_TOTAL_READING_TIME']].head(22)

In [ ]:
# --- MILESTONE 2: Now we want to calculate machine surprisal scores ---
import torch
import numpy as np
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Language models require heavy matrix multiplication
# We check if a GPU ("cuda") is available from our earlier environment setup
# Moving operations to a GPU makes inference significantly faster
# So, the lines below are for hardware acceleration
# Check for GPU from Cell 1
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running model on: {device}")

# Tokenizer converts the stimuli text (22 words above) into token ID numbers so that the model understands them
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Model is a pre-trained neural network model (GPT-2). Below, we push the model to the GPU
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

# Prepare inputs and move them to the GPU (two commands):
# We pass our Agatha Christie sentence into the tokenizer
# return_tensors="pt" tells it to return PyTorch ("pt" stands for PyTorch) tensors instead of standard Python lists
# This is the bridge between standard Python data and deep learning algorithms
inputs = tokenizer(sentence, return_tensors="pt")

# Why? GPT-2 is a PyTorch model (GPT2LMHeadModel)
# It only accepts PyTorch tensors as input
# If you just run tokenizer(sentence) without that argument, 
   # ... the Hugging Face tokenizer will process your text and return standard Python lists containing integers (the token IDs).
   # Example output: {'input_ids': [464, 4613, 2929, 286]}
# List outputs are useless for machine learning
   # Also, you cannot send a Python list to a GPU
# Using return_tensors="pt" tells tokenizer to wrap those numbers inside a PyTorch Tensor
   # Example output: {'input_ids': tensor([[464, 4613, 2929, 286]])}

# What is next:
# We must ensure our input data is on the exact same hardware chip as our model
# This loop moves all input tensors (like input_ids and attention_mask) to the device
inputs = {k: v.to(device) for k, v in inputs.items()}
# If not run, you may get an "Expected all tensors to be on the same device" error. The model and the data must be in the same place

# torch.no_grad() tells PyTorch we are testing, not training
# Disabling gradient tracking saves memory and speeds up execution
with torch.no_grad():
    outputs = model(**inputs)
    # Language models don't naturally spit out probabilities (0 to 1).
    # They output a vector of raw scores (logits) from −∞ to +∞ for all 50,000+ words in GPT-2's vocabulary 
    # So, 'logits' are the raw, unnormalized mathematical scores the model assigns to every single word in its vocabulary 
    # In the next block, we will convert these raw scores into probabilities to make sense of them
    logits = outputs.logits

# We will now calculate Softmax probabilities
probs = torch.nn.functional.softmax(logits[0], dim=-1)
# Softmax function converts raw logit scores into a valid probability distribution (all values sum to 1.0)
# dim=-1 means we apply Softmax across the very last dimension (the 50,000+ words in the vocabulary)

# The following line creates an empty list (we will use it later)
model_metrics = []

# Loop through tokens (Starting from second token due to 'First Word Problem')
# The loop variable i represents the Context (the word the model is currently looking at)
# The variable i+1 represents the Target (the word the model is trying to guess)
# The first word has the token number 0, so we start with it as the context
for i in range(len(inputs['input_ids'][0]) - 1):
    next_token_id = inputs['input_ids'][0][i+1]
    
    # Extract probability of the actual next word
    # The model guessed 50,000 probabilities for the next word
    # Below we extract the exact probability the model assigned to the next word
    # .item() pulls the number out of the PyTorch tensor and turns it into a standard float
    prob_next_token = probs[i, next_token_id].item()

    # Below is the core idea in Information Theory
       # If the model is 100% sure (prob=1), surprisal is 0 bits
       # If the model is highly surprised (prob=0.001), surprisal spikes
    surprisal = -np.log2(prob_next_token)

    # Why do we use Log Base 2 instead of natural log np.log?
        # Our resulting Surprisal metric is measured in bits
    # This ties NLP back to Claude Shannon's Information Theory
    
    # Convert the numerical ID back into human-readable text
    token_str = tokenizer.decode([next_token_id])
    
    model_metrics.append({
        'token': token_str, 
        'probability': prob_next_token,
        'surprisal': surprisal
    })

# Convert the list of dictionaries into a clean Pandas DataFrame
df_subwords = pd.DataFrame(model_metrics)
df_subwords.head(23)


In [ ]:
# If you set the head as 23 above, you will see classical NLP phenomena:
    # 1. The subword split in the last row ("subs" instead of "subsided")
        # We will use the aggregate_subwords function to resolve it
    # 2. Punctuation becomes tokens
        # The aggregate_subwords function will handle punctuation
    # 3. The proper-noun anomaly ("Styles")
        # This is not an error; it is a feature. Leave it in the data!

# --- Aligning BPE Subwords back into whole words ---
# We are interested in fixations on WHOLE words separated by spaces
# Language models process SUBWORDS 
# We must stitch the subwords back together so our datasets align

def aggregate_subwords(df):
    words = []                 # The empty bucket for our final merged words
    current_word = ""          # A temporary string to build the word
    current_surprisal = 0.0    # A temporary number to sum the surprisal
    
    for _, row in df.iterrows():
        token = row['token']
        
        # GPT-2's tokenizer marks the start of a new word by putting a space in front of it
        # In raw text, this is rendered as the special character 'Ġ'.
        if token.startswith(' ') or token.startswith('Ġ'):
            
            # 1. If we are already building a word, save it to the bucket
            if current_word:
                words.append({
                    'word': current_word.strip(), 
                    'surprisal': current_surprisal
                })
            
            # 2. Start building the NEW word
            current_word = token.strip()
            current_surprisal = row['surprisal']
            
        else:
            # Below is the merge command to handle subwords like "subs" + "ided"
            # If there is no space/'Ġ', it means this token is a piece of the current word
            # We glue the text together:
            current_word += token
            
            # We also add surprisal: This is fine because Surprisal = -log(Probability)
                # The probability of "subs" AND "ided" happening together is P(subs) * P(ided).
                # In log-space, multiplication becomes addition: log(A * B) = log(A) + log(B).
            current_surprisal += row['surprisal']

        #Punctuation does not have a space before it
        # The else block automatically glues the punctuation back onto the preceding word and adds their surprisals together
            
    # Save the very last word in the sentence!
    if current_word:
        words.append({'word': current_word.strip(), 'surprisal': current_surprisal})
        
    return pd.DataFrame(words)

# Run the function on the data
df_model = aggregate_subwords(df_subwords)
print("Subwords aligned! Ready for merge.")

In [ ]:
# --- MILESTONE 3: Bridging Fixations and Surprisal Scores ---
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Positional Alignment:
# Remember that we couldn't calculate surprisal for the very first word of the sentence (no context)
# We will drop the first word from the eye-tracking data, too
df_human_aligned = df_human.iloc[1:].reset_index(drop=True)

# Perform a direct "positional" merge by copying the array of values rather than pd.merge(on='word')
    # Some words, such as the article "the" may appear multiple times in a sentence

if len(df_human_aligned) == len(df_model):
    df_combined = df_human_aligned.copy()
    df_combined['surprisal'] = df_model['surprisal'].values
else:
    # Include a safety check when doing positional alignment
    raise ValueError(f"Alignment Error: Human words ({len(df_human_aligned)}) != Model words ({len(df_model)})")


# Let's use Spearman's Rho as the statistical test
# Why not Pearson (Linear): Reading times are not normally distributed
# Spearman's rank correlation handles outliers gracefully without assuming a normal distribution

correlation, p_value = spearmanr(df_combined['surprisal'], df_combined['WORD_TOTAL_READING_TIME'])
print(f"Spearman's rho: {correlation:.3f} | p-value: {p_value:.3e}\n")


# Below is visualization
plt.figure(figsize=(11, 7))

# regplot creates a scatter plot and fits an automatic linear regression line
# A positive slope visually proves that higher machine surprisal = higher reading times
sns.regplot(
    data=df_combined, x='surprisal', y='WORD_TOTAL_READING_TIME',
    scatter_kws={'alpha': 0.8, 'color': '#2b7bba', 's': 100},
    line_kws={'color': 'red', 'linewidth': 2}
)

# Loop through the dataframe to physically write the words next to their corresponding dots on the graph
# Adding +15 to Y pushes the text slightly above the dot
for i, txt in enumerate(df_combined['WORD']):
    plt.annotate(
        txt, 
        (df_combined['surprisal'].iloc[i], df_combined['WORD_TOTAL_READING_TIME'].iloc[i] + 15), 
        fontsize=10
    )

# Formatting the chart for presentation
plt.title('Cognitive Load vs. LM Predictability (GECO Corpus)')
plt.xlabel('GPT-2 Surprisal (bits)')
plt.ylabel('Human Total Reading Time (ms)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()